# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides an example for loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access and print dataset metadata
metadata = dataset.metadata
print(f"Name: {metadata.name}\nDescription: {metadata.description}\nPublished: {getattr(metadata, 'datePublished', 'N/A')}")

## 2. Data Overview
Review available record sets and fields referenced by their `@id` values. 

In [ ]:
# List record sets from the metadata
record_sets = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    if isinstance(metadata.recordSet, list):
        record_sets = [rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs for rs in metadata.recordSet]
    else:
        record_sets = [metadata.recordSet['@id'] if isinstance(metadata.recordSet, dict) and '@id' in metadata.recordSet else metadata.recordSet]
else:
    # If not available in metadata, attempt to introspect dataset
    # For mlcroissant, use dataset.record_sets if available
    try:
        record_sets = dataset.record_sets
    except AttributeError:
        record_sets = []

print("Available record sets:")
for rs_id in record_sets:
    print(f" - {rs_id}")

# For each record set, print example records and their field ids
for rs_id in record_sets:
    print(f"\nSample records for RecordSet '@id': {rs_id}")
    try:
        for i, record in enumerate(dataset.records(record_set=rs_id)):
            # Limit output for readability
            print(record)
            if i >= 1:
                break
    except Exception as e:
        print(f"Could not load records for {rs_id}: {e}")

## 3. Data Extraction
Load data from available record sets into DataFrames for analysis. All are referenced by their `@id`. Use the columns and fields identified above.

In [ ]:
# Extract data from each RecordSet
dataframes = {}

for rs_id in record_sets:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"\nLoaded DataFrame for RecordSet @id: {rs_id}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head())
    except Exception as e:
        print(f"Could not load records for {rs_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping data by attributes. Reference all fields by their `@id`.

In [ ]:
# Select a RecordSet and numeric field for EDA
main_rs_id = record_sets[0] if record_sets else None  # Use first record set as example
df = dataframes.get(main_rs_id, pd.DataFrame())
print(f"Using DataFrame for '@id': {main_rs_id}")

# Attempt to identify numeric columns
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print(f"Numeric fields: {numeric_fields}")

# For the sake of demonstration, pick the first numeric field
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    print(f"Using numeric field '@id': {numeric_field_id}")
    
    # Filter records above a threshold (here use mean as threshold)
    if df[numeric_field_id].notnull().any():
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())
        
        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized '{numeric_field_id}':")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a categorical field
        group_fields = [col for col in df.columns if pd.api.types.is_string_dtype(df[col])]
        if group_fields:
            group_field_id = group_fields[0]
            print(f"Grouping by field '@id': {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print("Grouped mean by category:")
            print(grouped_df.head())
else:
    print("No numeric fields available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between numeric and categorical fields.

In [ ]:
# Visualize numeric field distribution
if numeric_fields:
    plt.figure(figsize=(8,4))
    df[numeric_field_id].hist(bins=30)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

# Visualize grouped means (e.g. boxplot)
if 'group_field_id' in locals() and numeric_field_id:
    plt.figure(figsize=(8,4))
    df.boxplot(column=numeric_field_id, by=group_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.suptitle('')
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library. We loaded the metadata and records, reviewed available record sets and fields by their `@id`, extracted data into DataFrames, performed basic EDA, and visualized important metrics. 

This dataset provides a useful framework for AI-ready and reproducible data infrastructure, with survey-based insights into mental health trends in Kilifi County. Further analysis can build on this workflow by incorporating domain-specific statistical modeling or machine learning. 